In [2]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.2 MB/s eta 0:00:00


In [6]:
import os
from gensim.models import Word2Vec


def run_prediction():

    # ----------------------------------------------------------
    # 1. Configuration Based on the saved Skip-gram model file
    # ----------------------------------------------------------
    model_path = "iitj_skipgram.model"

    # ----------------------------------------------------------
    # 2. Pre-flight test is done-check whether the model file is on disk,
    # then trying to load it before we have our message straight
    # instead of mysterious Python exception by definition.
    # ----------------------------------------------------------
    if not os.path.exists(model_path):
        print(f"Error: {model_path} not found in the current directory.")
        print("Make sure you uploaded 'iitj_skipgram.model' (and any .npy files if created).")
        return  # Get out of the work; nothing can do without the model

    try:
        # ------------------------------------------------------
        # 3. Load disk model of Word2Vec.
        # Word2Vec.load() reinstates the entire model along with
        # weights, vocabulary and training settings
        # ------------------------------------------------------
        print(f"Loading model: {model_path}...")
        model = Word2Vec.load(model_path)
        print("Model loaded successfully.\n")


        # ======================================================
        # PART 1: Top-5 Nearest Neighbors
        # ======================================================
        # We retrieve 5 vocabulary words to each target word we retrieve
        # whose embedding vectors have the most similarity with each other, in terms of cosine similarity.
        # High similarity implies that the model learned such words
        # appear in similar contexts.
        # ======================================================

        target_words = ['research', 'student', 'phd', 'examinations', 'faculty']
        print("--- Nearest Neighbors ---")

        for word in target_words:
            word_lower = word.lower()  # Normalize case to correspond to training-time tokenization

            if word_lower in model.wv:
                # mostsimilar will give a list ofword,cosinescore tuples
                # topn=5 will only give the five nearest neighbours
                sims = model.wv.most_similar(word_lower, topn=5)

                print(f"Top 5 for '{word_lower}':")
                for n, s in sims:
                    # n = neighbour word, s = similarity score in range [-1, 1]
                    print(f"  -> {n} ({s:.4f})")
            else:
                # Filtering of words was done in training (below min_count threshold)
                # or never, merely, transpired in the corpus
                print(f"'{word_lower}' is not in the model's vocabulary.")

            print()  # Line of blankness between the word blocks to be read


        # ======================================================
        # PART 2: Analogy Experiments through Vector Arithmetic
        # ======================================================
        # Word2Vec allows analogical reasoning by means of simple
        # vector operations:
        #   most_similar(positive=[A, C], negative=[B])
        #   solves: B is to A as ? is to C
        #   by computing: vector(A) - vector(B) + vector(C)
        #   and finding the nearest vocabulary word to the result
        # ======================================================

        print("--- Analogy Experiments ---")
        # Format: UG : BTech :: PG : ?  => (BTech + PG) - UG

        try:
            # Create a symbol table of all the known words by the model
            # key_to_index associates words of the vocabulary with their integer indexes
            vocab = model.wv.key_to_index

            print("--- Running Analogy Experiments ---")

            # --------------------------------------------------
            # Analogy 1: Degree Hierarchy
            # Question: Undergraduate is to BTech as Postgraduate is to ?
            # Vector math: vector(btech) + vector(postgraduate) - vector(undergraduate)
            # --------------------------------------------------
            pos1 = ['btech', 'postgraduate']  # Vectors to add
            neg1 = ['undergraduate']           # Vector to subtract

            if all(w in vocab for w in pos1 + neg1):
                # topn=1 returns only the single best-matching word
                res1 = model.wv.most_similar(positive=pos1, negative=neg1, topn=1)
                print(f"1. [Undergraduate : BTech :: Postgraduate : ?] -> Result: {res1[0][0]}")
            else:
                # Inform the user which specific words caused the lookup to fail
                print("Degree analogy words missing. Check if 'btech', 'undergraduate', or 'postgraduate' are in your vocabulary.")

            # --------------------------------------------------
            # Analogy 2: Institutional Identity
            # Question: India is to Institute as IIT is to ?
            # Vector math: vector(institute) + vector(iit) - vector(indian)
            # --------------------------------------------------
            pos2 = ['institute', 'iit']  # Vectors to add
            neg2 = ['indian']             # Vector to subtract

            if all(w in vocab for w in pos2 + neg2):
                res2 = model.wv.most_similar(positive=pos2, negative=neg2, topn=1)
                print(f"2. [India : Institute :: IIT : ?] -> Result: {res2[0][0]}")
            else:
                print("Institutional analogy words missing. Check if 'institute', 'indian', or 'iit' are in your vocabulary.")

            # --------------------------------------------------
            # Analogy 3: Discipline to Administrative Unit
            # Question: Biology is to Science as Computing is to ?
            # Vector math: vector(science) + vector(computing) - vector(biology)
            # --------------------------------------------------
            pos3 = ['science', 'computing']  # Adding the category and the specific field
            neg3 = ['biology']               # Subtracting the unrelated field

            if all(w in vocab for w in pos3 + neg3):
                res3 = model.wv.most_similar(positive=pos3, negative=neg3, topn=1)
                print(f"3. [Biology : Science :: Computing : ?] -> Result: {res3[0][0]}")
            else:
                # Look for 'computing' or 'computer' depending on your scraper results
                print("Discipline analogy words missing. Check for 'science', 'computing', or 'biology'.")

            # --------------------------------------------------
            # Analogy 4: Roles and Responsibilities
            # Question: Student is to Study as Faculty is to ? (Expected: Research, Teaching, or Professor)
            # Vector math: vector(study) + vector(faculty) - vector(student)
            # --------------------------------------------------
            pos3 = ['study', 'faculty']  # High-level actions and roles
            neg3 = ['student']           # Subtly subtracts the 'learner' aspect

            if all(w in vocab for w in pos3 + neg3):
                res3 = model.wv.most_similar(positive=pos3, negative=neg3, topn=1)
                print(f"3. [Student : Study :: Faculty : ?] -> Result: {res3[0][0]}")
            else:
                # Error handling specifically for the Faculty-Research analogy
                print("Faculty analogy words missing. Check for 'study', 'faculty', or 'student'.")

        except Exception as e:
            # Identifies any unforeseen problem in the analogy computation,
            # e.g. bad vocabulary or internal Gensim mistakes.
            print(f"Analogy error: {e}")

    except EOFError:
        # EOFError normally implies that the.model file was saved incompletely
        # or was corrupted - user has to restart Task 2 to re-generate it
        print("Error: The model file is corrupted or truncated. Please re-save it in Task 2.")

    except Exception as e:
        # Fallback of any other runtime errors during model loading
        print(f"An unexpected error occurred: {e}")


# ----------------------------------------------------------
# Entry point — ensures run_prediction() only executes when
# this file is run directly, not when imported as a module
# ----------------------------------------------------------
if __name__ == "__main__":
    run_prediction()

Loading model: iitj_skipgram.model...
Model loaded successfully.

--- Nearest Neighbors ---
Top 5 for 'research':
  -> current (0.6181)
  -> ‘optics (0.5887)
  -> india. (0.5873)
  -> photonics’. (0.5865)
  -> learning; (0.5845)

Top 5 for 'student':
  -> student. (0.7187)
  -> necessary (0.6988)
  -> assistantship (0.6925)
  -> facilities. (0.6907)
  -> purpose, (0.6903)

Top 5 for 'phd':
  -> 1994 (0.8705)
  -> 2011 (0.8632)
  -> temperature. (0.8578)
  -> award. (0.8564)
  -> 2004, (0.8551)

Top 5 for 'examinations':
  -> minutes (0.9072)
  -> due (0.8927)
  -> long (0.8920)
  -> (three (0.8861)
  -> 20.1 (0.8840)

Top 5 for 'faculty':
  -> members (0.7149)
  -> world. (0.6806)
  -> member (0.6631)
  -> members, (0.6622)
  -> looking (0.6543)

--- Analogy Experiments ---
--- Running Analogy Experiments ---
1. [Undergraduate : BTech :: Postgraduate : ?] -> Result: nit
2. [India : Institute :: IIT : ?] -> Result: jodhpur
3. [Biology : Science :: Computing : ?] -> Result: outside
3. [S